# Lab 4: Markov Chain and Q-Learning

## Markov Chain
* **Shift from one state to another**
* **Depend on next step not sequence of past step.**
* **Memoryless**

### Properties
1. **State space**
2. **Transition Probability Matrix**
3. **Stationary distribution**
   * No change probability distribution.

## Markov Decision Process (MDP)
>> Sequential Decision making problem where outcome of an action is uncertain.

>> Components: State (S), Action (A), transition (P), Reward function (R), $\gamma$ (discount Rate)

## Q-Learning
* Trial and Error algorithm
* Learn the long term value
* Specific action in specific situation
* **Q** $\rightarrow$ Quality
* Lookup table (Q-table)
* *If state "S" and take action "A", How good will my total future pay off?*

### Mathematical Equation
$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ R + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

Where:
* $Q(s,a)$ (old value) $\rightarrow$ agent current think action a is worth
* $\alpha$ (learning rate) $\rightarrow$ "0" and "1" [0: nothing learn, 1: Learn new]
* $R$ (immediate Reward) $\rightarrow$ penalty score
* $\gamma$ (Discount factor) $\rightarrow$ "0" and "1" [reward against immediate one]
* $s'$ $\rightarrow$ Next state
* $\max_{a'} Q(s', a')$ $\rightarrow$ Best Expected future value

## Example: 4x4 Grid World

**Grid:**
| S0 | S1 | S2 | S3 |
|---|---|---|---|
| S4 | **S5** | S6 | S7 |
| S8 | S9 | **S10**| S11|
| S12| S13| S14| **S15**|

* **Start State:** S0
* **Goal State:** S15
* **Obstacles:** S5 & S10
* **Parameters:** $\alpha = 0.5$, $\gamma = 0.9$
* **Rewards:** 
  * $R = +10$ (Goal)
  * $R = -10$ (obstacle)
  * $R = -1$ (Normal)
* **Actions:** $\uparrow$ (U), $\downarrow$ (D), $\leftarrow$ (L), $\rightarrow$ (R)

### Initial Step:
**Q-table:** Intialized with zeros for all states ($S$) and actions.

### Episode Walkthrough

**Ep 1:**
* $S0 \xrightarrow{R} S1$
  * $Q(S0, R) = 0 + 0.5[-1 + 0.9(0) - 0] = -0.5$
* $S1 \xrightarrow{D} S5$ *(obstacle)*
  * $Q(S1, D) = 0 + 0.5[-10 + 0.9(0) - 0] = -5$

**Ep 2:**
* $S0 \xrightarrow{D} S4$
  * $Q(S0, D) = -0.5$
* ... *(skipping intermediate steps)* ...
* $S14 \xrightarrow{R} S15$ *(goal)*
  * $Q(S14, R) = 0 + 0.5[10 + 0.9(0) - 0] = 5$

**Ep 3 (Back propagate):**
* $Q(S14, R) = 5$
* $S13 \xrightarrow{R} S14$
  * Assume $Q(S13, R)$ was previously $-0.5$
  * $Q(S13, R) \leftarrow -0.5 + 0.5[-1 + 0.9(5) - (-0.5)]$
  * $= -0.5 + 0.5[-1 + 4.5 + 0.5] = -0.5 + 0.5[4] = -0.5 + 2 = 1.5$

## Code Implementation
Below is the Python code simulating the Q-Learning process on this 4x4 Grid Environment.

In [1]:
import numpy as np

# -------------------------------
# 4x4 Grid World
# -------------------------------

rows, cols = 4, 4
num_states = rows * cols
actions = ["UP", "DOWN", "LEFT", "RIGHT"]

alpha = 0.5       # Learning rate
gamma = 0.9       # Discount factor
episodes = 3

# Rewards
goal = 15
obstacles = [5, 10]

Q = np.zeros((num_states, 4))

# -------------------------------
# Helper Functions
# -------------------------------

def next_state(state, action):
    r = state // cols
    c = state % cols

    if action == 0 and r > 0:       # UP
        r -= 1
    elif action == 1 and r < rows-1: # DOWN
        r += 1
    elif action == 2 and c > 0:      # LEFT
        c -= 1
    elif action == 3 and c < cols-1: # RIGHT
        c += 1

    return r * cols + c


def reward(state):
    if state == goal:
        return 10
    elif state in obstacles:
        return -10
    else:
        return -1


def print_grid(agent):
    print("\nGrid:")

    for i in range(num_states):
        if i == agent:
            print(" A ", end="")
        elif i == goal:
            print(" G ", end="")
        elif i in obstacles:
            print(" X ", end="")
        else:
            print(f"{i:2d} ", end="")

        if (i + 1) % cols == 0:
            print()


def print_qtable():
    print("\nQ-Table")
    print("-----------------------------------------------")
    print("State\tUP\tDOWN\tLEFT\tRIGHT")
    print("-----------------------------------------------")

    for s in range(num_states):
        print(f"S{s}\t", end="")
        for a in range(4):
            print(f"{Q[s][a]:6.2f}", end="\t")
        print()
    print("-----------------------------------------------")


# -------------------------------
# Q-Learning
# -------------------------------

for ep in range(episodes):

    state = 0
    step = 1

    print("\n========================================")
    print(f"EPISODE {ep+1}")
    print("========================================")

    while state != goal:

        # Valid actions
        possible = []

        r = state // cols
        c = state % cols

        if r > 0:
            possible.append(0)
        if r < rows-1:
            possible.append(1)
        if c > 0:
            possible.append(2)
        if c < cols-1:
            possible.append(3)

        # Random action
        action = np.random.choice(possible)

        new_state = next_state(state, action)
        rwd = reward(new_state)

        # Q-learning update
        Q[state][action] = Q[state][action] + alpha * (
            rwd + gamma * np.max(Q[new_state]) - Q[state][action]
        )

        print(f"\nStep {step}")
        print(f"Current State : S{state}")
        print(f"Action        : {actions[action]}")
        print(f"Next State    : S{new_state}")
        print(f"Reward        : {rwd}")

        print_grid(new_state)
        print_qtable()

        # Stop if obstacle
        if new_state in obstacles:
            print("Obstacle Hit! Episode Ends.")
            break

        state = new_state
        step += 1

print("\nTraining Completed!")

print("\nFinal Q-Table")
print_qtable()



EPISODE 1

Step 1
Current State : S0
Action        : RIGHT
Next State    : S1
Reward        : -1

Grid:
 0  A  2  3 
 4  X  6  7 
 8  9  X 11 
12 13 14  G 

Q-Table
-----------------------------------------------
State	UP	DOWN	LEFT	RIGHT
-----------------------------------------------
S0	  0.00	  0.00	  0.00	 -0.50	
S1	  0.00	  0.00	  0.00	  0.00	
S2	  0.00	  0.00	  0.00	  0.00	
S3	  0.00	  0.00	  0.00	  0.00	
S4	  0.00	  0.00	  0.00	  0.00	
S5	  0.00	  0.00	  0.00	  0.00	
S6	  0.00	  0.00	  0.00	  0.00	
S7	  0.00	  0.00	  0.00	  0.00	
S8	  0.00	  0.00	  0.00	  0.00	
S9	  0.00	  0.00	  0.00	  0.00	
S10	  0.00	  0.00	  0.00	  0.00	
S11	  0.00	  0.00	  0.00	  0.00	
S12	  0.00	  0.00	  0.00	  0.00	
S13	  0.00	  0.00	  0.00	  0.00	
S14	  0.00	  0.00	  0.00	  0.00	
S15	  0.00	  0.00	  0.00	  0.00	
-----------------------------------------------

Step 2
Current State : S1
Action        : DOWN
Next State    : S5
Reward        : -10

Grid:
 0  1  2  3 
 4  A  6  7 
 8  9  X 11 
12 13 14  G 



## Conclusion and Discussions

**Conclusion:**
In this lab, we successfully explored the concepts of Markov Chains, Markov Decision Processes (MDPs), and the Q-Learning algorithm. Through a practical 4x4 grid example, we manually traced the trial-and-error approach, examining how an agent updates its Q-table using the Bellman equation. By interacting with the environment, receiving penalties for obstacles (-10) and normal moves (-1), and securing a reward (+10) upon reaching the goal, the agent dynamically learns the optimal path from the start state to the goal.

**Discussions:**
1. **Exploration vs. Exploitation:** The trial-and-error nature of Q-Learning highlights the balance required between exploring new paths and exploiting known high-reward paths. In our programmatic implementation, we employ an $\epsilon$-greedy strategy to handle this.
2. **Impact of Hyperparameters:**
    - **Learning Rate ($\alpha=0.5$):** Determines how much new information overrides old information. A value of 0.5 allows the agent to steadily integrate recent experiences.
    - **Discount Factor ($\gamma=0.9$):** A value close to 1 indicates that the agent cares significantly about long-term rewards, which is critical in a grid maze where the immediate reward is usually negative until the final goal is reached.
3. **Memoryless Property:** As part of the Markov property, the agent evaluates the next action strictly based on the current state (like S13) without needing the history of how it arrived there.
4. **Value Backpropagation:** Our manual trace (Ep 3) clearly demonstrated how the high reward received at the goal state (S15) backpropagates to previous states (S14 $\rightarrow$ S13). This allows the entire Q-table to eventually reflect the true "quality" of actions from any state in the grid.